# 🏥 AI Healthcare Assistant Platform
## Phase 5: FastAPI Backend — REST API
**Skills:** Python · FastAPI · SQL · ML · CV · Chatbot

---
### What we build in this phase:
- FastAPI app wiring all 3 AI modules (ML + CV + Chatbot)
- Pydantic request/response schemas with validation
- Endpoints: `/patients`, `/predict-risk`, `/scan-image`, `/chat`, `/dashboard`
- Startup model loader (loads all pkl/pth files once)
- API testing with `requests` directly from Jupyter
- Auto-generated Swagger docs at `/docs`

## Step 1: Write the Full API to Disk

In [7]:
import os

api_code = '''
import os, pickle, sqlite3, io, base64
import numpy as np
import pandas as pd
import faiss
import cv2
import torch
import torch.nn as nn
from torchvision import models, transforms
from PIL import Image
from sentence_transformers import SentenceTransformer
from fastapi import FastAPI, UploadFile, File, HTTPException
from fastapi.middleware.cors import CORSMiddleware
from pydantic import BaseModel, Field
from typing import Optional, List
import uvicorn

DB_PATH   = "healthcare_ai/data/healthcare.db"
MODEL_DIR = "healthcare_ai/models"
IMG_DIR   = "healthcare_ai/uploads/images"
CLASSES   = ["normal", "pneumonia", "skin_lesion"]
FEATURES  = ["age","gender_enc","glucose","blood_pressure","bmi","insulin","heart_rate"]
DEVICE    = torch.device("cuda" if torch.cuda.is_available() else "cpu")

app = FastAPI(
    title="AI Healthcare Assistant API",
    description="Unified API: ML Risk Prediction + Computer Vision + Medical Chatbot",
    version="1.0.0"
)
app.add_middleware(
    CORSMiddleware,
    allow_origins=["*"], allow_methods=["*"], allow_headers=["*"]
)

class ModelStore:
    ml_models  = {}
    scaler     = None
    cv_model   = None
    embedder   = None
    faiss_index= None
    knowledge_base = []

store = ModelStore()

@app.on_event("startup")
async def load_models():
    print("Loading ML models...")
    for disease in ["diabetes", "heart_disease", "hypertension"]:
        path = os.path.join(MODEL_DIR, f"{disease}_model.pkl")
        with open(path, "rb") as f:
            store.ml_models[disease] = pickle.load(f)
    with open(os.path.join(MODEL_DIR, "scaler.pkl"), "rb") as f:
        store.scaler = pickle.load(f)

    print("Loading CV model...")
    cv = models.resnet18(weights=None)
    cv.fc = nn.Sequential(
        nn.Dropout(0.4), nn.Linear(cv.fc.in_features, 128),
        nn.ReLU(), nn.Linear(128, 3)
    )
    cv.load_state_dict(torch.load(
        os.path.join(MODEL_DIR, "cv_model.pth"), map_location=DEVICE
    ))
    cv.eval()
    store.cv_model = cv.to(DEVICE)

    print("Loading chatbot components...")
    store.embedder    = SentenceTransformer("all-MiniLM-L6-v2")
    store.faiss_index = faiss.read_index(os.path.join(MODEL_DIR, "medical_faiss.index"))
    with open(os.path.join(MODEL_DIR, "knowledge_base.pkl"), "rb") as f:
        store.knowledge_base = pickle.load(f)
    print("All models loaded!")

class PatientCreate(BaseModel):
    name:       str
    age:        int  = Field(..., ge=1, le=120)
    gender:     str  = Field(..., pattern="^(Male|Female|Other)$")
    email:      str
    blood_type: str

class VitalsInput(BaseModel):
    patient_id:     int
    age:            int
    gender:         str
    glucose:        float = Field(..., ge=50,  le=400)
    blood_pressure: float = Field(..., ge=40,  le=200)
    bmi:            float = Field(..., ge=10,  le=60)
    insulin:        float = Field(..., ge=0,   le=500)
    heart_rate:     int   = Field(..., ge=30,  le=200)

class ChatMessage(BaseModel):
    patient_id: int  = 1
    message:    str

def preprocess_image_bytes(img_bytes):
    nparr = np.frombuffer(img_bytes, np.uint8)
    img   = cv2.imdecode(nparr, cv2.IMREAD_COLOR)
    img   = cv2.resize(img, (224, 224))
    lab   = cv2.cvtColor(img, cv2.COLOR_BGR2LAB)
    l,a,b = cv2.split(lab)
    clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8,8))
    lab   = cv2.merge([clahe.apply(l), a, b])
    img   = cv2.cvtColor(cv2.cvtColor(lab, cv2.COLOR_LAB2BGR), cv2.COLOR_BGR2RGB)
    tfm   = transforms.Compose([
        transforms.ToTensor(),
        transforms.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225])
    ])
    return tfm(Image.fromarray(img)).unsqueeze(0).to(DEVICE)

def rag_retrieve(query, top_k=2):
    vec = store.embedder.encode([query], convert_to_numpy=True)
    vec = vec / np.linalg.norm(vec, axis=1, keepdims=True)
    scores, indices = store.faiss_index.search(vec.astype(np.float32), top_k)
    return [(store.knowledge_base[i], float(s)) for i,s in zip(indices[0], scores[0]) if i>=0]

def db_execute(query, params=()):
    with sqlite3.connect(DB_PATH) as conn:
        conn.execute(query, params)

def db_fetch(query, params=()):
    with sqlite3.connect(DB_PATH) as conn:
        return pd.read_sql(query, conn, params=params)

@app.get("/")
def root():
    return {"status": "online", "message": "AI Healthcare Assistant API v1.0"}

@app.get("/health")
def health_check():
    return {
        "ml_models_loaded":  len(store.ml_models) == 3,
        "cv_model_loaded":   store.cv_model is not None,
        "chatbot_loaded":    store.embedder is not None,
        "faiss_vectors":     store.faiss_index.ntotal if store.faiss_index else 0
    }

@app.post("/patients")
def create_patient(data: PatientCreate):
    db_execute(
        "INSERT OR IGNORE INTO patients (name,age,gender,email,blood_type) VALUES (?,?,?,?,?)",
        (data.name, data.age, data.gender, data.email, data.blood_type)
    )
    patient = db_fetch("SELECT * FROM patients WHERE email=?", (data.email,))
    return {"status": "created", "patient": patient.to_dict(orient="records")[0]}

@app.get("/patients/{patient_id}")
def get_patient(patient_id: int):
    patient = db_fetch("SELECT * FROM patients WHERE id=?", (patient_id,))
    if patient.empty:
        raise HTTPException(status_code=404, detail="Patient not found")
    vitals = db_fetch("SELECT * FROM vitals WHERE patient_id=? ORDER BY recorded_at DESC LIMIT 1", (patient_id,))
    preds  = db_fetch("SELECT * FROM risk_predictions WHERE patient_id=? ORDER BY predicted_at DESC LIMIT 3", (patient_id,))
    return {
        "patient": patient.to_dict(orient="records")[0],
        "latest_vitals": vitals.to_dict(orient="records"),
        "recent_predictions": preds.to_dict(orient="records")
    }

@app.post("/predict-risk")
def predict_risk(data: VitalsInput):
    gender_enc = 1 if data.gender == "Female" else 0
    row = np.array([[data.age, gender_enc, data.glucose,
                     data.blood_pressure, data.bmi, data.insulin, data.heart_rate]])
    row_scaled = store.scaler.transform(row)
    results = []
    for disease, model in store.ml_models.items():
        prob  = float(model.predict_proba(row_scaled)[0][1])
        level = "high" if prob > 0.7 else ("medium" if prob > 0.4 else "low")
        results.append({"disease": disease, "risk_score": round(prob,3), "risk_level": level})
        db_execute(
            "INSERT INTO risk_predictions (patient_id,disease,risk_score,risk_level,model_version) VALUES (?,?,?,?,?)",
            (data.patient_id, disease, round(prob,3), level, "v1.0")
        )
    overall = max(results, key=lambda x: x["risk_score"])
    return {"patient_id": data.patient_id, "predictions": results, "highest_risk": overall}

@app.post("/scan-image")
async def scan_image(patient_id: int = 1, file: UploadFile = File(...)):
    img_bytes = await file.read()
    tensor    = preprocess_image_bytes(img_bytes)
    with torch.no_grad():
        probs = torch.softmax(store.cv_model(tensor), dim=1)[0].cpu().numpy()
    pred_idx   = int(np.argmax(probs))
    diagnosis  = CLASSES[pred_idx]
    confidence = round(float(probs[pred_idx]), 4)
    img_path   = os.path.join(IMG_DIR, f"upload_p{patient_id}_{file.filename}")
    with open(img_path, "wb") as f:
        f.write(img_bytes)
    db_execute(
        "INSERT INTO image_scans (patient_id,image_path,scan_type,cv_diagnosis,confidence) VALUES (?,?,?,?,?)",
        (patient_id, img_path, "upload", diagnosis, confidence)
    )
    return {"patient_id": patient_id, "diagnosis": diagnosis, "confidence": confidence,
            "all_probabilities": {c: round(float(p),4) for c,p in zip(CLASSES, probs)}}

@app.post("/chat")
def chat(data: ChatMessage):
    text = data.message.lower()
    emergency_kw = ["dying","cant breathe","heart attack","stroke","not breathing","seizure"]
    urgent_kw    = ["chest pain","severe","worst","sudden","vomiting blood"]
    if any(k in text for k in emergency_kw):
        intent = "emergency"
    elif any(k in text for k in urgent_kw):
        intent = "urgent"
    elif any(w in text for w in ["hello","hi ","hey"]):
        intent = "greeting"
    else:
        intent = "symptom_inquiry"

    if intent == "greeting":
        response = "Hello! I am your AI Healthcare Assistant. Please describe your symptoms."
        severity = "low"
    else:
        retrieved = rag_retrieve(data.message)
        if retrieved:
            best, score = retrieved[0]
            severity = "emergency" if intent == "emergency" else best["severity"]
            actions  = {"emergency":"CALL 911 IMMEDIATELY","high":"Visit ER within 2 hours",
                        "medium":"See doctor within 48 hours","low":"Monitor at home"}
            response = f"Assessment: {best[\'condition\']}. Advice: {best[\'advice\']}. Action: {actions.get(severity)}"
        else:
            severity = "low"
            response = "Please describe your symptoms in more detail."

    db_execute(
        "INSERT INTO symptom_logs (patient_id,symptoms_text,bot_response,severity) VALUES (?,?,?,?)",
        (data.patient_id, data.message, response, severity)
    )
    return {"patient_id": data.patient_id, "intent": intent, "response": response, "severity": severity}

@app.get("/dashboard")
def dashboard():
    stats = {
        "total_patients":    int(db_fetch("SELECT COUNT(*) as n FROM patients")["n"][0]),
        "total_predictions": int(db_fetch("SELECT COUNT(*) as n FROM risk_predictions")["n"][0]),
        "total_scans":       int(db_fetch("SELECT COUNT(*) as n FROM image_scans")["n"][0]),
        "total_chats":       int(db_fetch("SELECT COUNT(*) as n FROM symptom_logs")["n"][0]),
        "avg_risk_score":    round(float(db_fetch("SELECT AVG(risk_score) as a FROM risk_predictions")["a"][0]),3),
        "high_risk_count":   int(db_fetch("SELECT COUNT(*) as n FROM risk_predictions WHERE risk_level=\'high\'")["n"][0]),
    }
    high_risk = db_fetch(
        """SELECT p.name, p.age, r.disease, r.risk_score, r.risk_level
           FROM patients p JOIN risk_predictions r ON p.id=r.patient_id
           WHERE r.risk_level=\'high\' ORDER BY r.risk_score DESC LIMIT 5"""
    ).to_dict(orient="records")
    return {"summary": stats, "top_high_risk_patients": high_risk}

if __name__ == "__main__":
    uvicorn.run("main:app", host="0.0.0.0", port=8000, reload=False)
'''

os.makedirs('healthcare_ai/api', exist_ok=True)
with open('healthcare_ai/api/main.py', 'w') as f:
    f.write(api_code.strip())

print('✅ API written to healthcare_ai/api/main.py')

✅ API written to healthcare_ai/api/main.py


## Step 2: Start the API Server (background thread)

In [9]:
import subprocess, time, sys

# Launch uvicorn in background
server = subprocess.Popen(
    [sys.executable, "-m", "uvicorn",
     "healthcare_ai.api.main:app",
     "--host", "0.0.0.0",
     "--port", "8000",
     "--reload"],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT
)

print('⏳ Starting API server... (waiting 8 seconds for models to load)')
time.sleep(8)
print('✅ Server should be running at http://localhost:8000')
print('📖 Swagger docs at     http://localhost:8000/docs')

⏳ Starting API server... (waiting 8 seconds for models to load)
✅ Server should be running at http://localhost:8000
📖 Swagger docs at     http://localhost:8000/docs


## Step 3: Test All Endpoints with requests

In [11]:
import requests, json

BASE = 'http://localhost:8000'

def pretty(label, resp):
    print(f'\n{"═"*55}')
    print(f'🔹 {label}  [{resp.status_code}]')
    print('═'*55)
    try:
        print(json.dumps(resp.json(), indent=2))
    except:
        print(resp.text)

# 1. Health check
pretty('GET /', requests.get(f'{BASE}/'))

# 2. Health status
pretty('GET /health', requests.get(f'{BASE}/health'))


═══════════════════════════════════════════════════════
🔹 GET /  [200]
═══════════════════════════════════════════════════════
{
  "status": "online",
  "message": "AI Healthcare Assistant API v1.0"
}

═══════════════════════════════════════════════════════
🔹 GET /health  [200]
═══════════════════════════════════════════════════════
{
  "ml_models_loaded": true,
  "cv_model_loaded": true,
  "chatbot_loaded": true,
  "faiss_vectors": 20
}


In [13]:
# 3. Create a new patient
new_patient = {
    'name': 'Sarah Johnson',
    'age': 42,
    'gender': 'Female',
    'email': 'sarah.johnson@example.com',
    'blood_type': 'A+'
}
pretty('POST /patients', requests.post(f'{BASE}/patients', json=new_patient))


═══════════════════════════════════════════════════════
🔹 POST /patients  [200]
═══════════════════════════════════════════════════════
{
  "status": "created",
  "patient": {
    "id": 51,
    "name": "Sarah Johnson",
    "age": 42,
    "gender": "Female",
    "email": "sarah.johnson@example.com",
    "blood_type": "A+",
    "created_at": "2026-06-02 16:34:10"
  }
}


In [15]:
# 4. ML Risk Prediction
vitals = {
    'patient_id': 1,
    'age': 52, 'gender': 'Male',
    'glucose': 175.0, 'blood_pressure': 130.0,
    'bmi': 34.2, 'insulin': 190.0,
    'heart_rate': 98
}
pretty('POST /predict-risk', requests.post(f'{BASE}/predict-risk', json=vitals))


═══════════════════════════════════════════════════════
🔹 POST /predict-risk  [200]
═══════════════════════════════════════════════════════
{
  "patient_id": 1,
  "predictions": [
    {
      "disease": "diabetes",
      "risk_score": 0.82,
      "risk_level": "high"
    },
    {
      "disease": "heart_disease",
      "risk_score": 0.82,
      "risk_level": "high"
    },
    {
      "disease": "hypertension",
      "risk_score": 0.938,
      "risk_level": "high"
    }
  ],
  "highest_risk": {
    "disease": "hypertension",
    "risk_score": 0.938,
    "risk_level": "high"
  }
}


In [17]:
# 5. Chatbot
messages = [
    {'patient_id': 1, 'message': 'Hello!'},
    {'patient_id': 1, 'message': 'I have severe chest pain and left arm numbness'},
    {'patient_id': 2, 'message': 'I have been very thirsty and urinating frequently'},
]
for msg in messages:
    pretty(f'POST /chat — "{msg["message"][:40]}"',
           requests.post(f'{BASE}/chat', json=msg))


═══════════════════════════════════════════════════════
🔹 POST /chat — "Hello!"  [200]
═══════════════════════════════════════════════════════
{
  "patient_id": 1,
  "intent": "greeting",
  "response": "Hello! I am your AI Healthcare Assistant. Please describe your symptoms.",
  "severity": "low"
}

═══════════════════════════════════════════════════════
🔹 POST /chat — "I have severe chest pain and left arm nu"  [200]
═══════════════════════════════════════════════════════
{
  "patient_id": 1,
  "intent": "urgent",
  "response": "Assessment: Possible cardiac event (heart attack). Advice: EMERGENCY: Call 911 immediately. Chew aspirin if not allergic. Do not drive yourself.. Action: CALL 911 IMMEDIATELY",
  "severity": "emergency"
}

═══════════════════════════════════════════════════════
🔹 POST /chat — "I have been very thirsty and urinating f"  [200]
═══════════════════════════════════════════════════════
{
  "patient_id": 2,
  "intent": "symptom_inquiry",
  "response": "Assessment: P

In [19]:
# 6. Dashboard
pretty('GET /dashboard', requests.get(f'{BASE}/dashboard'))


═══════════════════════════════════════════════════════
🔹 GET /dashboard  [200]
═══════════════════════════════════════════════════════
{
  "summary": {
    "total_patients": 51,
    "total_predictions": 56,
    "total_scans": 3,
    "total_chats": 62,
    "avg_risk_score": 0.579,
    "high_risk_count": 22
  },
  "top_high_risk_patients": [
    {
      "name": "Tyler Anderson",
      "age": 69,
      "disease": "diabetes",
      "risk_score": 0.965,
      "risk_level": "high"
    },
    {
      "name": "Erica Jones",
      "age": 75,
      "disease": "hypertension",
      "risk_score": 0.949,
      "risk_level": "high"
    },
    {
      "name": "Kenneth Cantrell",
      "age": 25,
      "disease": "diabetes",
      "risk_score": 0.943,
      "risk_level": "high"
    },
    {
      "name": "Julie Young",
      "age": 46,
      "disease": "diabetes",
      "risk_score": 0.942,
      "risk_level": "high"
    },
    {
      "name": "Todd Luna",
      "age": 21,
      "disease": "hyperten

In [21]:
# 7. Get patient profile
pretty('GET /patients/1', requests.get(f'{BASE}/patients/1'))


═══════════════════════════════════════════════════════
🔹 GET /patients/1  [200]
═══════════════════════════════════════════════════════
{
  "patient": {
    "id": 1,
    "name": "Todd Luna",
    "age": 21,
    "gender": "Male",
    "email": "urose@example.com",
    "blood_type": "AB+",
    "created_at": "2026-06-02 15:57:45"
  },
  "latest_vitals": [
    {
      "id": 1,
      "patient_id": 1,
      "glucose": 101.8,
      "blood_pressure": 71.2,
      "bmi": 20.3,
      "insulin": 150.7,
      "heart_rate": 89,
      "recorded_at": "2026-06-02 15:57:45"
    }
  ],
  "recent_predictions": [
    {
      "id": 54,
      "patient_id": 1,
      "disease": "diabetes",
      "risk_score": 0.82,
      "risk_level": "high",
      "model_version": "v1.0",
      "predicted_at": "2026-06-02 16:34:18"
    },
    {
      "id": 55,
      "patient_id": 1,
      "disease": "heart_disease",
      "risk_score": 0.82,
      "risk_level": "high",
      "model_version": "v1.0",
      "predicted_at": "202

## Step 4: Test CV endpoint with a real image

In [23]:
import cv2, numpy as np

# Generate a test pneumonia image and send to API
def make_test_image():
    base = np.random.randint(155, 185, (224, 224, 3), dtype=np.uint8)
    for _ in range(5):
        cx, cy = np.random.randint(40, 184, 2)
        cv2.ellipse(base, (cx,cy), (30,20), 0, 0, 360, (230,230,230), -1)
    return base

test_img = make_test_image()
_, buf   = cv2.imencode('.png', cv2.cvtColor(test_img, cv2.COLOR_RGB2BGR))
img_bytes = buf.tobytes()

resp = requests.post(
    f'{BASE}/scan-image?patient_id=1',
    files={'file': ('test_scan.png', img_bytes, 'image/png')}
)
pretty('POST /scan-image (pneumonia test)', resp)


═══════════════════════════════════════════════════════
🔹 POST /scan-image (pneumonia test)  [200]
═══════════════════════════════════════════════════════
{
  "patient_id": 1,
  "diagnosis": "pneumonia",
  "confidence": 0.9344,
  "all_probabilities": {
    "normal": 0.016,
    "pneumonia": 0.9344,
    "skin_lesion": 0.0496
  }
}


## Step 5: Stop server + Summary

In [25]:
server.terminate()
print('🛑 Server stopped.')

print("""
╔══════════════════════════════════════════════════╗
║        Phase 5 Complete — API Summary            ║
╠══════════════════════════════════════════════════╣
║  GET  /              Health check                ║
║  GET  /health        Model load status           ║
║  POST /patients      Register patient            ║
║  GET  /patients/{id} Full patient profile        ║
║  POST /predict-risk  ML disease risk scores      ║
║  POST /scan-image    CV medical image diagnosis  ║
║  POST /chat          RAG medical chatbot         ║
║  GET  /dashboard     Platform analytics          ║
╠══════════════════════════════════════════════════╣
║  Swagger UI: http://localhost:8000/docs          ║
╚══════════════════════════════════════════════════╝
""")

🛑 Server stopped.

╔══════════════════════════════════════════════════╗
║        Phase 5 Complete — API Summary            ║
╠══════════════════════════════════════════════════╣
║  GET  /              Health check                ║
║  GET  /health        Model load status           ║
║  POST /patients      Register patient            ║
║  GET  /patients/{id} Full patient profile        ║
║  POST /predict-risk  ML disease risk scores      ║
║  POST /scan-image    CV medical image diagnosis  ║
║  POST /chat          RAG medical chatbot         ║
║  GET  /dashboard     Platform analytics          ║
╠══════════════════════════════════════════════════╣
║  Swagger UI: http://localhost:8000/docs          ║
╚══════════════════════════════════════════════════╝



In [1]:
# Fix chatbot intent in main.py
import os

with open("healthcare_ai/api/main.py", "r", encoding="utf-8") as f:
    content = f.read()

old_chat = '''    if intent == "greeting":
        response = "Hello! I am your AI Healthcare Assistant. Please describe your symptoms."
        severity = "low"
    else:
        retrieved = rag_retrieve(data.message)
        if retrieved:
            best, score = retrieved[0]
            severity = "emergency" if intent == "emergency" else best["severity"]
            actions  = {"emergency":"CALL 911 IMMEDIATELY","high":"Visit ER within 2 hours",
                        "medium":"See doctor within 48 hours","low":"Monitor at home"}
            response = f"Assessment: {best['condition']}. Advice: {best['advice']}. Action: {actions.get(severity)}"
        else:
            severity = "low"
            response = "Please describe your symptoms in more detail."'''

new_chat = '''    # Short or non-medical messages
    short_msg = len(data.message.strip().split()) <= 3
    medical_keywords = ["pain","fever","headache","cough","breath","bleed","dizzy",
                        "nausea","vomit","chest","heart","skin","rash","sugar",
                        "glucose","pressure","fatigue","tired","sore","throat",
                        "stomach","back","arm","leg","swelling","cold","flu"]
    has_medical = any(k in data.message.lower() for k in medical_keywords)

    if intent == "greeting" or (short_msg and not has_medical):
        greetings = ["hello","hi","hey","ok","okay","good","fine","thanks",
                     "thank","yes","no","sure","great","nice","cool","wow"]
        if any(g in data.message.lower() for g in greetings):
            response = "Hello! I am your AI Healthcare Assistant. Please describe your symptoms in detail and I will help assess them."
        else:
            response = "I am here to help with medical symptom assessment. Please describe what you are experiencing."
        severity = "low"
    elif not has_medical and not intent in ["emergency","urgent"]:
        response = "I can help with medical symptom assessment. Please describe any symptoms you are experiencing such as pain, fever, cough, or other health concerns."
        severity = "low"
    else:
        retrieved = rag_retrieve(data.message)
        if retrieved:
            best, score = retrieved[0]
            # Only use retrieved result if similarity is high enough
            if score > 0.3:
                severity = "emergency" if intent == "emergency" else best["severity"]
                actions  = {"emergency":"CALL 911 IMMEDIATELY","high":"Visit ER within 2 hours",
                            "medium":"See doctor within 48 hours","low":"Monitor at home"}
                response = f"Assessment: {best['condition']}. Advice: {best['advice']}. Action: {actions.get(severity)}"
            else:
                severity = "low"
                response = "I am not sure about your symptoms. Please describe them in more detail."
        else:
            severity = "low"
            response = "Please describe your symptoms in more detail so I can help you better."'''

content = content.replace(old_chat, new_chat)

with open("healthcare_ai/api/main.py", "w", encoding="utf-8") as f:
    f.write(content)

print("Chatbot fixed!")

Chatbot fixed!


In [ ]:
import subprocess, sys, os, time

project_root = r"C:\Users\kssud\AI Healthcare Assistant platform"
os.chdir(project_root)

server = subprocess.Popen(
    [sys.executable, "-m", "uvicorn",
     "healthcare_ai.api.main:app",
     "--host", "0.0.0.0", "--port", "8000"],
    cwd=project_root,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True
)

# Wait and capture output
time.sleep(15)
output = ""
while True:
    line = server.stdout.readline()
    if not line:
        break
    output += line
    print(line.strip())

INFO:     Started server process [29040]
INFO:     Waiting for application startup.
Loading ML models...
Loading CV model...
Loading chatbot...

Loading weights: 100%|##########| 103/103 [00:00<00:00, 4903.17it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  |
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  |

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://0.0.0.0:8000 (Press CTRL+C to quit)
All models loaded!
INFO:     127.0.0.1:51324 - "GET /app HTTP/1.1" 200 OK
INFO:     127.0.0.1:51324 - "GET /dashboard HTTP/1.1" 200 OK
INFO:     127.0.0.1:51324 - "GET /docs HTTP/1.1" 200 OK
INFO:     127.0.0.1:51324 - "GET /openapi.json HTTP/1.1" 200 OK
INFO:     127.0.0.1:51324 - "GET /dashboard HTTP/1.1" 200 OK
INFO:     127.0.0.1:59748 - "POST /chat HTT

In [ ]:
import requests

r = requests.get("http://localhost:8000/health")
print("Server:", r.json())

---
## ✅ Phase 5 Complete!

**What you built:**
- FastAPI app with 8 production-ready endpoints
- Single startup model loader (ML + CV + Chatbot all in memory)
- Pydantic validation on all inputs
- Full CORS support for mobile frontend
- All results persisted to SQLite database
- Tested every endpoint from Jupyter with `requests`

**Next → Phase 6: Mobile-Ready Frontend + Cloud Deployment (AWS/GCP)**
- React responsive UI with patient dashboard
- Dockerize the entire app
- Deploy to AWS ECS or GCP Cloud Run
- Environment variables + production config